# ДЗ №2. Задание 3
### Выполнил: Лёвин В.А.
### Группа: ИУ6-54Б

Активность по времени.


## Подготовка

Используем поездки за апрель 2026 года. Отправления считаем по времени начала, прибытия — по времени завершения. Прибытия 1 мая не входят в расчёт апреля. События без ID станции пропускаем.

In [16]:
import os
import folium
from folium.plugins import HeatMapWithTime
from IPython.display import display
from pyspark import SparkConf
from pyspark.sql import SparkSession, functions as F

conf = SparkConf().setMaster("local[2]").setAppName("HW2-Time")
spark = SparkSession.builder.config(conf=conf).getOrCreate()

trips = spark.read.csv("../*citibike-tripdata*.csv", header=True)
dates = trips.select(F.to_date("started_at").alias("date")).distinct()
day_count = dates.count()

starts = trips.selectExpr(
    "start_station_id as station_id", "start_station_name as station_name",
    "cast(start_lat as double) as latitude", "cast(start_lng as double) as longitude",
    "cast(started_at as timestamp) as time", "1 as starts", "0 as ends"
)
ends = trips.selectExpr(
    "end_station_id as station_id", "end_station_name as station_name",
    "cast(end_lat as double) as latitude", "cast(end_lng as double) as longitude",
    "cast(ended_at as timestamp) as time", "0 as starts", "1 as ends"
)
events = (
    starts.unionByName(ends).filter("station_id != ''")
    .withColumn("date", F.to_date("time"))
    .withColumn("period", F.floor(F.hour("time") / 6))
    .join(dates, "date")
)
stations = events.groupBy("station_id").agg(
    F.first("station_name", ignorenulls=True).alias("station_name"),
    F.avg("latitude").alias("latitude"), F.avg("longitude").alias("longitude")
).cache()
daily_counts = events.groupBy("station_id", "date", "period").agg(
    F.sum("starts").alias("starts"), F.sum("ends").alias("ends")
).cache()

## 1. Среднее количество отправлений и прибытий в день

Суммы по станции делим на 30 дней, включая дни без поездок. В выводе — первые 10 станций, значения округлены до двух знаков.

In [17]:
daily_averages = daily_counts.groupBy("station_id").agg(
    (F.sum("starts") / day_count).alias("starts"),
    (F.sum("ends") / day_count).alias("ends")
).join(stations, "station_id")

daily_averages.select(
    "station_id", "station_name",
    F.round("starts", 2).alias("starts"), F.round("ends", 2).alias("ends")
).orderBy("station_id").show(10, truncate=False)

+----------+---------------------------+------+-----+
|station_id|station_name               |starts|ends |
+----------+---------------------------+------+-----+
|1234.56   |Morgan HCT Charging        |9.4   |10.67|
|1964.01   |101 St & Fort Hamilton Pkwy|1.5   |1.5  |
|2009.04   |Shore Rd & 4 Ave           |3.83  |3.87 |
|2042.01   |4 Ave & 99 St              |3.03  |3.13 |
|2086.07   |99 St & 3 Ave              |1.77  |1.87 |
|2118.13   |Fort Hamilton Pkwy & 95 St |1.4   |1.5  |
|2127.03   |Shore Rd & 97 St           |3.43  |3.5  |
|2158.08   |4 Ave & 94 St              |2.97  |4.17 |
|2190.08   |Fort Hamilton Pkwy & 92 St |4.4   |4.43 |
|2200.07   |96 St & Marine Ave         |2.97  |3.07 |
+----------+---------------------------+------+-----+
only showing top 10 rows



## 2. Среднее по времени суток

Ночь -  00:00–05:59 

Утро - 06:00–11:59

День - 12:00–17:59

Вечер - 18:00–23:59

Отсутствующие сочетания станции и периода заполняем нулями

In [18]:
periods = spark.sql("""
    SELECT * FROM VALUES
    (0, 'Ночь 00:00–05:59'), (1, 'Утро 06:00–11:59'),
    (2, 'День 12:00–17:59'), (3, 'Вечер 18:00–23:59')
    AS periods(period, period_name)
""")
period_names = [row.period_name for row in periods.orderBy("period").collect()]

period_averages = daily_counts.groupBy("station_id", "period").agg(
    (F.sum("starts") / day_count).alias("starts"),
    (F.sum("ends") / day_count).alias("ends")
)
period_averages = (
    stations.crossJoin(periods)
    .join(period_averages, ["station_id", "period"], "left")
    .fillna(0, subset=["starts", "ends"])
)
period_averages.orderBy("station_id", "period").select(
    "station_id", "station_name", "period_name",
    F.round("starts", 2).alias("starts"), F.round("ends", 2).alias("ends")
).show(12)

+----------+--------------------+-----------------+------+----+
|station_id|        station_name|      period_name|starts|ends|
+----------+--------------------+-----------------+------+----+
|   1234.56| Morgan HCT Charging| Ночь 00:00–05:59|  0.37| 0.7|
|   1234.56| Morgan HCT Charging| Утро 06:00–11:59|   4.0|3.67|
|   1234.56| Morgan HCT Charging| День 12:00–17:59|  2.23|2.47|
|   1234.56| Morgan HCT Charging|Вечер 18:00–23:59|   2.8|3.83|
|   1964.01|101 St & Fort Ham...| Ночь 00:00–05:59|   0.0| 0.0|
|   1964.01|101 St & Fort Ham...| Утро 06:00–11:59|  0.23|0.13|
|   1964.01|101 St & Fort Ham...| День 12:00–17:59|  0.77|0.87|
|   1964.01|101 St & Fort Ham...|Вечер 18:00–23:59|   0.5| 0.5|
|   2009.04|    Shore Rd & 4 Ave| Ночь 00:00–05:59|  0.13| 0.2|
|   2009.04|    Shore Rd & 4 Ave| Утро 06:00–11:59|  0.93|0.53|
|   2009.04|    Shore Rd & 4 Ave| День 12:00–17:59|   1.6|1.87|
|   2009.04|    Shore Rd & 4 Ave|Вечер 18:00–23:59|  1.17|1.27|
+----------+--------------------+-------

## 3. Среднее по периодам в среду и воскресенье

В апреле 5 сред и 4 воскресенья. В Spark воскресенье имеет номер 1, среда — 4. 

In [19]:
weekday_days = (
    dates.withColumn("weekday", F.dayofweek("date"))
    .groupBy("weekday").agg(F.count("*").alias("days"))
    .filter("weekday IN (1, 4)")
    .withColumn("day_name", F.when(F.col("weekday") == 1, "Воскресенье").otherwise("Среда"))
)

weekday_totals = (
    daily_counts.withColumn("weekday", F.dayofweek("date"))
    .filter("weekday IN (1, 4)")
    .groupBy("station_id", "period", "weekday")
    .agg(F.sum("starts").alias("starts"), F.sum("ends").alias("ends"))
)
weekday_averages = (
    stations.crossJoin(periods).crossJoin(weekday_days)
    .join(weekday_totals, ["station_id", "period", "weekday"], "left")
    .fillna(0, subset=["starts", "ends"])
    .withColumn("starts", F.col("starts") / F.col("days"))
    .withColumn("ends", F.col("ends") / F.col("days"))
)
weekday_averages.orderBy("station_id", "weekday", "period").select(
    "station_id", "station_name", "day_name", "period_name",
    F.round("starts", 2).alias("starts"), F.round("ends", 2).alias("ends")
).show(weekday_averages.count())

+------------+--------------------+-----------+-----------------+------+------+
|  station_id|        station_name|   day_name|      period_name|starts|  ends|
+------------+--------------------+-----------+-----------------+------+------+
|     1234.56| Morgan HCT Charging|Воскресенье| Ночь 00:00–05:59|   0.5|  0.75|
|     1234.56| Morgan HCT Charging|Воскресенье| Утро 06:00–11:59|   2.0|   4.0|
|     1234.56| Morgan HCT Charging|Воскресенье| День 12:00–17:59|   3.5|  1.75|
|     1234.56| Morgan HCT Charging|Воскресенье|Вечер 18:00–23:59|  2.25|  4.75|
|     1234.56| Morgan HCT Charging|      Среда| Ночь 00:00–05:59|   0.0|   0.2|
|     1234.56| Morgan HCT Charging|      Среда| Утро 06:00–11:59|   5.2|   4.6|
|     1234.56| Morgan HCT Charging|      Среда| День 12:00–17:59|   2.4|   2.4|
|     1234.56| Morgan HCT Charging|      Среда|Вечер 18:00–23:59|   3.2|   3.2|
|     1964.01|101 St & Fort Ham...|Воскресенье| Ночь 00:00–05:59|   0.0|   0.0|
|     1964.01|101 St & Fort Ham...|Воскр

## 4. Тепловые карты по времени суток

Показываем средние из пункта 2: сначала отправления, затем прибытия.

In [20]:
points = period_averages.select("period", "latitude", "longitude", "starts", "ends").collect()
maximum = max(max(point.starts, point.ends) for point in points)
os.makedirs("output", exist_ok=True)

for column, title in [("starts", "Отправления"), ("ends", "Прибытия")]:
    frames = [[], [], [], []]
    for point in points:
        if point[column] > 0:
            frames[point.period].append([
                point.latitude, point.longitude, point[column] / maximum
            ])

    time_map = folium.Map(location=[40.73, -73.98], zoom_start=11, tiles="OpenStreetMap")
    HeatMapWithTime(frames, index=period_names, name=title).add_to(time_map)
    folium.LayerControl().add_to(time_map)
    time_map.save(f"output/{column}_heatmap.html")
    display(time_map)

spark.catalog.clearCache()